In [0]:
%run "./common functions"

In [0]:
# from concurrent.futures import ThreadPoolExecutor, as_completed

# notebooks = [
#     "./LinkedIn/getting linkedin jobs",
#     "./workday/workday_final"
# ]

# def run_notebook(path):
#     try:
#         result = dbutils.notebook.run(path, 0)
#         return path, result
#     except:
#         print(f"Error running notebook: {path}")

# with ThreadPoolExecutor(max_workers=len(notebooks)) as executor:
#     futures = [executor.submit(run_notebook, nb) for nb in notebooks]

#     for future in as_completed(futures):
#         notebook, result = future.result()
#         print(f"{notebook} completed. Result: {result}")

In [0]:
df = spark.read.json("/Volumes/raw_catalogue/jobs/vol/jobs_from_github/")
df.createOrReplaceTempView("source_vw")

In [0]:
%sql
create or replace table staging_catalogue.jobs.all_jobs as
select * from (
select title, company, url, description, location from source_vw
union 
select title, company, url, description, location from raw_catalogue.jobs.linkedJobs
union
select title, company, url, description, location from staging_catalogue.jobs.workday
) a
left anti join main_catalogue.jobs.all_jobs b
on a.url = b.url

In [0]:
scored_df = (spark.sql(f"""select ai_query("databricks-llama-4-maverick", concat('{p1}', 'Company Name:', company, description, '{p2}', '{format}', '{rules}')) as cl,* from staging_catalogue.jobs.all_jobs""")
             .withColumn("score", get_json_object("cl", "$.score").cast("int")) 
            .withColumn("company_type", get_json_object("cl", "$.company_type"))
            .drop("cl", "description", "source"))

In [0]:
filtered_df = scored_df.filter(scored_df.score >= 70).orderBy(col("score").desc())
filtered_df.write.mode("overwrite").saveAsTable("staging_catalogue.jobs.naukari_linkedin")

In [0]:
%sql
select * from staging_catalogue.jobs.naukari_linkedin

In [0]:
form_message_and_send("staging_catalogue.jobs.naukari_linkedin")

In [0]:
%sql
insert into main_catalogue.jobs.all_jobs
select * from staging_catalogue.jobs.naukari_linkedin